# 06 - Statistical Rigor Layer
Point estimates alone overstate certainty. This notebook adds: bootstrap confidence intervals, hypothesis tests with effect sizes, Kaplan-Meier survival curves and a Cox proportional-hazards model. Helpers live in `src/stats_utils.py`.

In [1]:
import sys, warnings; sys.path.append('..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_confint
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from src import eda, stats_utils as U
pd.set_option('display.width', 200); sns.set_theme(style='whitegrid')
df = eda.load()
seg = pd.read_csv('../data/segments.csv'); df = df.merge(seg[['customerID', 'segment']], on='customerID')
pred = pd.read_csv('../data/test_predictions.csv')
y = pred.churn.values; p_lr = pred.p_logreg.values; p_hgb = pred.p_hgb.values
len(df), len(pred)

(7043, 1409)

## 1. Bootstrap confidence intervals on model performance
2,000 resamples of the 1,409 test customers (percentile method). The paired version resamples the *same* customers for both models, so the comparison is not distorted by test-set luck.

In [2]:
B = 2000
rows = []
for name, p in [('logreg', p_lr), ('hgb', p_hgb)]:
    for metric, fn in [('ROC-AUC', roc_auc_score), ('PR-AUC', average_precision_score)]:
        bs = U.bootstrap_metric(y, p, fn, B)
        lo, hi = U.percentile_ci(bs)
        rows.append({'model': name, 'metric': metric, 'point': round(fn(y, p), 3), 'CI 2.5%': round(lo, 3), 'CI 97.5%': round(hi, 3)})
auc_ci = pd.DataFrame(rows); print(auc_ci.to_string(index=False))
d_roc = U.bootstrap_paired_diff(y, p_hgb, p_lr, roc_auc_score, B); d_pr = U.bootstrap_paired_diff(y, p_hgb, p_lr, average_precision_score, B)
print('\nPaired bootstrap, HGB minus LR:')
print('  ROC-AUC diff %.4f, 95%% CI (%.4f, %.4f)' % ((d_roc.mean(),) + U.percentile_ci(d_roc)))
print('  PR-AUC  diff %.4f, 95%% CI (%.4f, %.4f)' % ((d_pr.mean(),) + U.percentile_ci(d_pr)))
bs_roc = U.bootstrap_metric(y, p_lr, roc_auc_score, B)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(bs_roc, bins=40, color='#4C78A8'); ax[0].axvline(roc_auc_score(y, p_lr), c='k'); [ax[0].axvline(v, c='r', ls='--') for v in U.percentile_ci(bs_roc)]
ax[0].set(title='Bootstrap distribution of test ROC-AUC (logreg)', xlabel='ROC-AUC')
ax[1].hist(d_roc, bins=40, color='#E4572E'); ax[1].axvline(0, c='k'); [ax[1].axvline(v, c='r', ls='--') for v in U.percentile_ci(d_roc)]
ax[1].set(title='Paired difference in ROC-AUC (HGB - logreg)', xlabel='difference')
plt.tight_layout(); plt.savefig('../reports/figures/13_bootstrap_auc.png', dpi=150); plt.show()

 model  metric  point  CI 2.5%  CI 97.5%
logreg ROC-AUC  0.839    0.815     0.860
logreg  PR-AUC  0.628    0.573     0.680
   hgb ROC-AUC  0.840    0.816     0.862
   hgb  PR-AUC  0.659    0.606     0.707



Paired bootstrap, HGB minus LR:
  ROC-AUC diff 0.0011, 95% CI (-0.0082, 0.0104)
  PR-AUC  diff 0.0301, 95% CI (0.0069, 0.0535)


**Reading it.**
- Logistic regression test ROC-AUC is **0.839, 95% CI (0.815, 0.860)**: with a test set this size the true AUC could plausibly be anywhere in a ~0.045-wide band, which is the honest way to quote it.
- **ROC-AUC: no difference between the models** (paired diff +0.001, CI -0.008 to +0.010 includes zero).
- **PR-AUC: gradient boosting is ahead by about 0.03 (CI +0.007 to +0.054, excludes zero).** This refines the Phase 5 reading: on this test split the gap is statistically detectable, though it is small and cross-validation showed a much smaller gap (0.664 vs 0.659). I keep logistic regression as the primary model for interpretability, with gradient boosting as a defensible choice if precision at low recall is the priority.

In [3]:
rows = []
idx = U.boot_idx(len(y), B)
vals = np.array([U.top_frac_stats(y[i], p_lr[i], 0.2) for i in idx])
pt = U.top_frac_stats(y, p_lr, 0.2)
for j, nm in enumerate(['precision in top 20%', 'recall (share of churners reached)', 'lift']):
    lo, hi = U.percentile_ci(vals[:, j]); rows.append({'metric': nm, 'point': round(pt[j], 3), 'CI 2.5%': round(lo, 3), 'CI 97.5%': round(hi, 3)})
pd.DataFrame(rows)

,metric,point,CI 2.5%,CI 97.5%
0,precision in top 20%,0.665,0.605,0.719
1,recall (share of churners reached),0.500,0.457,0.540
2,lift,2.507,2.291,2.705


**Top-20% targeting with uncertainty:** precision **66.5% (CI 60.5-71.9)**, share of churners reached **50.0% (CI 45.7-54.0)**, lift **2.51x (CI 2.29-2.71)**. Even the lower bounds are well above random (26.5% precision, 20% recall, 1.0x lift), so the targeting benefit is not a fluke of this particular split.

## 2. Bootstrap CIs on segment churn rates
Percentile bootstrap, cross-checked against Wilson intervals (analytic).

In [4]:
order = df.groupby('segment').Churn.mean().sort_values(ascending=False).index
rows = []
for s in order:
    v = df.loc[df.segment == s, 'Churn'].values
    bs = np.array([v[i].mean() for i in U.boot_idx(len(v), B)])
    lo, hi = U.percentile_ci(bs); wlo, whi = proportion_confint(v.sum(), len(v), method='wilson')
    rows.append({'segment': s, 'n': len(v), 'churn_rate': round(v.mean(), 4), 'boot 2.5%': round(lo, 4), 'boot 97.5%': round(hi, 4), 'Wilson 2.5%': round(wlo, 4), 'Wilson 97.5%': round(whi, 4)})
segci = pd.DataFrame(rows); print(segci.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.errorbar(segci.churn_rate * 100, range(len(segci)), xerr=[(segci.churn_rate - segci['boot 2.5%']) * 100, (segci['boot 97.5%'] - segci.churn_rate) * 100], fmt='o', c='#E4572E', capsize=4)
ax.set_yticks(range(len(segci))); ax.set_yticklabels(segci.segment); ax.invert_yaxis(); ax.set(xlabel='Churn rate (%) with 95% bootstrap CI', title='Segment churn rates with uncertainty')
plt.tight_layout(); plt.savefig('../reports/figures/14_segment_ci.png', dpi=150); plt.show()

                      segment    n  churn_rate  boot 2.5%  boot 97.5%  Wilson 2.5%  Wilson 97.5%
At-risk fiber, month-to-month 1957      0.5662     0.5447      0.5871       0.5441        0.5880
 Mid-tier DSL, month-to-month 1507      0.2760     0.2548      0.2999       0.2541        0.2992
                Loyal premium 2037      0.1139     0.1006      0.1281       0.1008        0.1284
            Budget phone-only 1542      0.0733     0.0597      0.0869       0.0613        0.0874


The four intervals do not overlap and are narrow (about +/-2 percentage points), so the ordering of segments is not in doubt. Bootstrap and Wilson intervals agree to the second decimal, a useful sanity check. The Phase 4 claim that the at-risk fiber segment churns at **56.6%** comes with a 95% CI of roughly **54.5%-58.7%**.

## 3. Hypothesis tests: statistical vs practical significance
Two-proportion z-tests between segments, each with the **difference and its CI, relative risk and Cohen's h** (h of ~0.2 is small, ~0.5 medium, ~0.8 large). Gender is included as a **negative control**: there is no reason for it to matter, so the test should (and does) show nothing. An overall chi-square across the four segments is below.

In [5]:
def two_prop(a, b, la, lb):
    xa, na = a.sum(), len(a); xb, nb = b.sum(), len(b)
    z, pval = proportions_ztest([xa, xb], [na, nb])
    lo, hi = confint_proportions_2indep(xa, na, xb, nb, method='wald')
    pa, pb = xa / na, xb / nb
    return {'comparison': f'{la} vs {lb}', 'rate_A': round(pa, 3), 'rate_B': round(pb, 3), 'diff (pp)': round(100 * (pa - pb), 1),
            '95% CI diff (pp)': f'({100*lo:.1f}, {100*hi:.1f})', 'relative risk': round(pa / pb, 2), "Cohen's h": round(U.cohens_h(pa, pb), 2),
            'z': round(z, 2), 'p-value': f'{pval:.2g}'}
S = lambda name: df.loc[df.segment == name, 'Churn']
res = [two_prop(S('At-risk fiber, month-to-month'), S('Mid-tier DSL, month-to-month'), 'At-risk fiber', 'Mid-tier DSL'),
       two_prop(S('Mid-tier DSL, month-to-month'), S('Loyal premium'), 'Mid-tier DSL', 'Loyal premium'),
       two_prop(S('Loyal premium'), S('Budget phone-only'), 'Loyal premium', 'Budget phone-only'),
       two_prop(df.loc[df.gender == 'Female', 'Churn'], df.loc[df.gender == 'Male', 'Churn'], 'Female', 'Male (negative control)')]
tests = pd.DataFrame(res); print(tests.to_string(index=False))
ct = pd.crosstab(df.segment, df.Churn); chi2, pv, dof, _ = stats.chi2_contingency(ct)
V = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"\nOverall chi-square across 4 segments: chi2={chi2:.1f}, dof={dof}, p={'<1e-300' if pv < 1e-300 else format(pv, '.2g')}, Cramer's V={V:.2f}")

                        comparison  rate_A  rate_B  diff (pp) 95% CI diff (pp)  relative risk  Cohen's h     z p-value
     At-risk fiber vs Mid-tier DSL   0.566   0.276       29.0     (25.9, 32.2)           2.05       0.60 17.05 3.2e-65
     Mid-tier DSL vs Loyal premium   0.276   0.114       16.2     (13.6, 18.9)           2.42       0.42 12.35 5.1e-35
Loyal premium vs Budget phone-only   0.114   0.073        4.1       (2.2, 6.0)           1.55       0.14  4.08 4.6e-05
 Female vs Male (negative control)   0.269   0.262        0.8      (-1.3, 2.8)           1.03       0.02  0.72    0.47

Overall chi-square across 4 segments: chi2=1440.8, dof=3, p=<1e-300, Cramer's V=0.45


**Statistical vs practical significance.**
- With 7,000 customers, even modest differences give tiny p-values, so *p-values alone are uninformative here*. The effect sizes carry the message.
- **At-risk fiber vs mid-tier DSL:** +29 percentage points, relative risk 2.05, h = 0.60. Both significant and practically large: the retention budget should go here.
- **Loyal premium vs budget phone-only:** p = 4.6e-5 (statistically significant) but the gap is **4 percentage points with h = 0.14, a small effect**. Both groups are low-churn, so the distinction would not change any retention decision. This is the statistically-significant-but-practically-modest case.
- **Female vs male:** 26.9% vs 26.2%, p = 0.47, h = 0.02: no effect, as the negative control should show.
- Overall association between segment and churn: chi-square 1,441 on 3 dof, **Cramer's V = 0.45**, a strong association.
- **Caveats.** Four pairwise tests: a Bonferroni threshold of 0.0125 leaves every conclusion unchanged. The segments were learned from the same customers, so these tests describe the data in hand rather than confirm segments on fresh data; churn was never used to form the clusters, which limits (but does not remove) the circularity.

## 4. Survival analysis: Kaplan-Meier
Treating each customer's `tenure` as time-to-event, with `Churn=1` as the event and active customers as right-censored. The 11 tenure-0 customers are excluded (a duration of zero cannot enter the model), leaving 7,032 customers, 1,869 events and 5,163 censored.

In [6]:
sv = df[df.tenure > 0].copy()          # tenure 0 -> duration 0, cannot enter a survival model (11 customers)
T, E = sv.tenure, sv.Churn
kmf = KaplanMeierFitter().fit(T, E, label='All customers')
print('n =', len(sv), '| events =', E.sum(), '| censored =', (1 - E).sum())
print('Median survival (months):', kmf.median_survival_time_)
print('S(t) at 12/24/48/72 months:', kmf.survival_function_at_times([12, 24, 48, 72]).round(3).values)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for c in ['Month-to-month', 'One year', 'Two year']:
    m = sv.Contract == c; KaplanMeierFitter().fit(T[m], E[m], label=c).plot_survival_function(ax=ax[0], ci_show=True)
ax[0].set(title='Kaplan-Meier: retention by contract type', xlabel='Tenure (months)', ylabel='P(still a customer)', ylim=(0, 1.02))
for s in order:
    m = sv.segment == s; KaplanMeierFitter().fit(T[m], E[m], label=s).plot_survival_function(ax=ax[1], ci_show=False)
ax[1].set(title='Kaplan-Meier: retention by segment', xlabel='Tenure (months)', ylim=(0, 1.02))
plt.tight_layout(); plt.savefig('../reports/figures/15_kaplan_meier.png', dpi=150); plt.show()
rows = []
for grp, col in [('Contract', 'Contract'), ('Segment', 'segment')]:
    r = multivariate_logrank_test(sv.tenure, sv[col], sv.Churn); rows.append({'grouping': grp, 'log-rank chi2': round(r.test_statistic, 1), 'p-value': f'{r.p_value:.2g}'})
print(pd.DataFrame(rows).to_string(index=False))
tab = []
for c in ['Month-to-month', 'One year', 'Two year']:
    m = sv.Contract == c; k = KaplanMeierFitter().fit(T[m], E[m])
    tab.append({'contract': c, **{f'S({t}mo)': round(float(k.survival_function_at_times(t).iloc[0]), 3) for t in [6, 12, 24, 48]}, 'median': k.median_survival_time_})
pd.DataFrame(tab)

n = 7032 | events = 1869 | censored = 5163
Median survival (months): inf
S(t) at 12/24/48/72 months: [0.843 0.789 0.709 0.593]


grouping  log-rank chi2 p-value
Contract         2352.9       0
 Segment         2558.7       0


,contract,S(6mo),S(12mo),S(24mo),S(48mo),median
0,Month-to-month,0.786,0.703,0.586,0.397,35.0
1,One year,0.997,0.991,0.978,0.917,inf
2,Two year,1.000,1.000,1.000,0.996,inf


**Findings.**
- Overall retention: **84% survive 12 months, 79% 24 months, 71% 48 months, 59% 72 months.** The median is not reached (more than half of customers are still active at 72 months).
- **Month-to-month customers: 70% survive to 12 months, 59% to 24, 40% to 48; median lifetime is 35 months.** One-year contracts keep 99%/98%/92% at the same horizons, and two-year contracts essentially 100%.
- Log-rank tests reject equal survival across contract types (chi-square 2,353) and across segments (chi-square 2,559), both with p below 1e-300.
- **Caveats that matter.** (1) Two-year contracts cannot be cancelled early without penalty, so their near-perfect curve is partly structural. (2) Contract type is the contract *at snapshot*, treated as fixed over a customer's life. Customers who switched to a long contract had to survive until they switched, so the contract effect is probably overstated. A time-varying covariate would fix this, but the data has no contract history.
- **Read the right tails with care.** Beyond roughly 65 months few customers remain at risk, so late drops (for example the mid-tier DSL curve) rest on a handful of people. Quote survival at 12-48 months, not at the very end.

## 5. Cox proportional-hazards model
Hazard ratios (HR > 1 = faster churn) after adjusting for the other covariates. Two diagnostics drove the modelling choices below.

In [7]:
cx = sv[['tenure', 'Churn', 'Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen', 'Partner', 'Dependents',
         'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MonthlyCharges']].copy()
cx['MonthlyCharges'] = (cx.MonthlyCharges - cx.MonthlyCharges.mean()) / cx.MonthlyCharges.std()
for c in ['PaperlessBilling', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']:
    cx[c] = (cx[c] == 'Yes').astype(int)
cxd = pd.get_dummies(cx, columns=['Contract', 'InternetService', 'PaymentMethod'], drop_first=False)
cxd = cxd.drop(columns=['Contract_Month-to-month', 'InternetService_DSL', 'PaymentMethod_Bank transfer (automatic)']).astype(float)
# R^2 of MonthlyCharges explained by the service columns (collinearity diagnostic)
from sklearn.linear_model import LinearRegression
svc = cxd.drop(columns=['tenure', 'Churn', 'MonthlyCharges']); lm = LinearRegression().fit(svc, cxd.MonthlyCharges)
print('R^2 of MonthlyCharges explained by the other covariates:', round(lm.score(svc, cxd.MonthlyCharges), 3))
cxn = cxd.drop(columns='MonthlyCharges')
cph = CoxPHFitter().fit(cxn, 'tenure', 'Churn')
print('Concordance (unstratified, no MonthlyCharges):', round(cph.concordance_index_, 3))
ph = proportional_hazard_test(cph, cxn, time_transform='rank')
print('\nProportional-hazards test (Schoenfeld), smallest p-values:'); print(ph.summary.sort_values('p').head(6).round(4).to_string())

R^2 of MonthlyCharges explained by the other covariates: 0.999


Concordance (unstratified, no MonthlyCharges): 0.866



Proportional-hazards test (Schoenfeld), smallest p-values:
                   test_statistic       p  -log2(p)
Contract_One year         39.8252  0.0000   31.7455
MultipleLines             17.4960  0.0000   15.0840
Contract_Two year         16.6149  0.0000   14.4146
StreamingTV               14.0426  0.0002   12.4500
TechSupport               13.4575  0.0002   12.0007
OnlineSecurity            12.7721  0.0004   11.4729


**Diagnostic 1: `MonthlyCharges` is 99.9% determined by the service columns** (R-squared 0.999), because the monthly bill is essentially a sum of the services taken. Keeping it makes the model unstable (next cell: Fiber HR 1.86 with a CI of 0.60-5.71, MonthlyCharges HR CI 0.19-2.87). I drop it; the services carry its information.

**Diagnostic 2: the proportional-hazards assumption fails for `Contract`** (Schoenfeld test, p < 0.0001, the smallest p-values in the table above). Contract changes the *shape* of the hazard over time, not just its level, so a single hazard ratio would be wrong. The fix is to **stratify on Contract** (a separate baseline hazard per contract type) rather than estimate a coefficient for it.

In [8]:
# (a) with MonthlyCharges: show the instability
cs_a = sv[['tenure', 'Churn', 'Contract']].join(cxd.drop(columns=['tenure', 'Churn', 'Contract_One year', 'Contract_Two year']))
a = CoxPHFitter().fit(cs_a, 'tenure', 'Churn', strata=['Contract']).summary
print('WITH MonthlyCharges  -> Fiber HR %.2f (95%% CI %.2f-%.2f); MonthlyCharges HR %.2f (95%% CI %.2f-%.2f)' % (
    a.loc['InternetService_Fiber optic', 'exp(coef)'], a.loc['InternetService_Fiber optic', 'exp(coef) lower 95%'], a.loc['InternetService_Fiber optic', 'exp(coef) upper 95%'],
    a.loc['MonthlyCharges', 'exp(coef)'], a.loc['MonthlyCharges', 'exp(coef) lower 95%'], a.loc['MonthlyCharges', 'exp(coef) upper 95%']))
# (b) final model: stratified on Contract, MonthlyCharges removed
cs = sv[['tenure', 'Churn', 'Contract']].join(cxn.drop(columns=['tenure', 'Churn', 'Contract_One year', 'Contract_Two year']))
cph_s = CoxPHFitter().fit(cs, 'tenure', 'Churn', strata=['Contract'])
print('Concordance (stratified on Contract):', round(cph_s.concordance_index_, 3))
ph2 = proportional_hazard_test(cph_s, cs, time_transform='rank'); print(ph2.summary.sort_values('p').head(5).round(4).to_string())
hr = cph_s.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].rename(columns={'exp(coef)': 'HR', 'exp(coef) lower 95%': 'lo', 'exp(coef) upper 95%': 'hi'}).sort_values('HR')
sig = hr[hr.p < 0.05]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.errorbar(sig.HR, range(len(sig)), xerr=[sig.HR - sig.lo, sig.hi - sig.HR], fmt='o', c='#4C78A8', capsize=3); ax.axvline(1, c='k', lw=1)
ax.set_yticks(range(len(sig))); ax.set_yticklabels(sig.index); ax.set_xscale('log'); ax.set(xlabel='Hazard ratio (log scale, 95% CI)', title='Cox model (stratified by contract): significant effects on churn hazard')
plt.tight_layout(); plt.savefig('../reports/figures/16_cox_hazard_ratios.png', dpi=150); plt.show()
hr.round(3).sort_values('HR', ascending=False)

WITH MonthlyCharges  -> Fiber HR 1.86 (95% CI 0.60-5.71); MonthlyCharges HR 0.75 (95% CI 0.19-2.87)


Concordance (stratified on Contract): 0.72
                 test_statistic       p  -log2(p)
TechSupport             14.9433  0.0001   13.1399
StreamingTV             14.4178  0.0001   12.7377
MultipleLines           13.6703  0.0002   12.1643
StreamingMovies         12.9202  0.0003   11.5871
OnlineSecurity          11.3335  0.0008   10.3594


,HR,lo,hi,p
covariate,,,,
PaymentMethod_Mailed check,1.784,1.501,2.121,0.000
PaymentMethod_Electronic check,1.783,1.550,2.051,0.000
InternetService_Fiber optic,1.458,1.265,1.680,0.000
PaperlessBilling,1.200,1.075,1.341,0.001
PhoneService,1.088,0.893,1.326,0.401
StreamingTV,0.960,0.863,1.068,0.456
SeniorCitizen,0.941,0.842,1.051,0.279
Dependents,0.938,0.820,1.073,0.353
PaymentMethod_Credit card (automatic),0.919,0.769,1.098,0.351


**Reading the stratified model.**
- Raising hazard: **fiber optic (HR 1.46, CI 1.27-1.68)**, **electronic check (1.78)**, **mailed check (1.78)** relative to bank transfer, and paperless billing (1.20).
- Lowering hazard: **no internet (0.35)**, **online security (0.53)**, **online backup (0.53)**, **having a partner (0.61)**, **multiple lines (0.64)**, **tech support (0.66)**, device protection (0.74).
- **Streaming TV/movies are not risk factors here** (HR 0.96 and 0.89), unlike the Phase 5 logistic odds ratios (1.4x). The difference traces to `MonthlyCharges`: the logistic model kept it alongside the service flags, so the streaming coefficients absorbed collinear price information. The Cox result is the more trustworthy one.
- The mailed-check hazard ratio is higher than its raw churn rate (19%) would suggest, because hazard ratios are adjusted for tenure and the other covariates. I treat it as a hypothesis to check rather than a headline finding.
- **Concordance is 0.72 (stratified).** The unstratified 0.87 is inflated because contract type almost fixes who can churn early; the stratified value measures ranking *within* contract type. Neither is comparable to the logistic model's AUC.
- **Remaining limitation:** after stratifying, several service covariates (tech support, streaming, multiple lines) still fail the Schoenfeld test at p < 0.001. I did not quantify how large those departures are, so read the hazard ratios as time-averaged effects and not as exact constants.

## Takeaways
- Test ROC-AUC **0.839 (95% CI 0.815-0.860)**; top-20% targeting reaches **50% of churners (CI 45.7-54.0%) at 2.5x lift (CI 2.3-2.7)**.
- Segment differences are large in practical terms (h up to 0.60), not just statistically significant; the negative control (gender) is null.
- Survival analysis: month-to-month median customer lifetime is **35 months**; the same drivers (fiber, payment method, no protective services) surface in a different model family.
- Statistical hygiene found two real problems (collinear price, non-proportional contract hazard) and fixed them rather than reporting a misleading model.